In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args
import skopt.plots as skplt
import matplotlib.pyplot as plt

# Import your existing pipeline functions
from RaTag.io.file_ops import iter_waveforms
from RaTag.el_tpc.waveform_features import find_s1, find_s2, compute_timing_statistics

# 1. Define your representative dataset
# Assume `selected_sets` is a pre-loaded list of 24 SetPmt objects 
# (4 EL runs * 6 drift sets).
selected_sets = [...] 
MAX_FRAMES = 1000 # Strict limit per set as requested

# 2. Define the Hyperparameter Search Space
# We are optimizing 'N' (sigma multiplier) rather than raw voltage.
space = [
    Real(2.0, 10.0, name='N_s1'),          # S1 Threshold multiplier
    Real(1.5, 8.0, name='N_s2'),           # S2 Threshold multiplier
    Integer(5, 50, name='window_size'),    # S2 Smoothing window
    Real(0.005, 0.05, name='threshold_bs'),# Baseline suppression
    Real(0.1, 0.7, name='t_drift_margin')  # Margin for t_min
]

# 3. The Objective Function
@use_named_args(space)
def objective(N_s1, N_s2, window_size, threshold_bs, t_drift_margin):
    total_loss = 0.0
    valid_sets = 0
    
    for set_pmt in selected_sets:
        # Context extraction: get the noise floor for this specific set
        # Assuming set_pmt has an attribute for baseline noise sigma. 
        # If not, you'd calculate this from the first few waveforms.
        noise_sigma = set_pmt.noise_sigma if hasattr(set_pmt, 'noise_sigma') else 0.05 
        
        # Convert N to dynamic physical thresholds
        threshold_s1 = noise_sigma * N_s1
        threshold_s2 = noise_sigma * N_s2
        
        t_drift_margin_time = (set_pmt.time_drift or 0.0) * t_drift_margin
        
        # Buffers for this set
        out_s1, out_s2_start = [], []
        s1_anchor = -5.0
        
        for wf in iter_waveforms(set_pmt, max_files=MAX_FRAMES):
            t_s1 = find_s1(wf, threshold=threshold_s1, t_max=-2.5)
            if not np.all(np.isnan(t_s1)):
                s1_anchor = np.nanmean(t_s1)
                
            t_s2_st, t_s2_end = find_s2(
                wf, 
                threshold_s2=threshold_s2, 
                t_min=s1_anchor + t_drift_margin_time, 
                window_size=int(window_size), 
                threshold_bs=threshold_bs
            )
            
            out_s1.append(t_s1)
            out_s2_start.append(t_s2_st)
            
        # Compute Stats
        s1_arr = np.concatenate(out_s1)
        s2_arr = np.concatenate(out_s2_start)
        
        stats_s1 = compute_timing_statistics(s1_arr, name="t_s1")
        stats_s2 = compute_timing_statistics(s2_arr, name="t_s2_start")
        
        std_s1 = stats_s1.get("t_s1_std", 0.0)
        std_s2 = stats_s2.get("t_s2_start_std", 0.0)
        
        # PENALTY LOGIC: If standard deviation is 0, it means detection failed 
        # (no signals found). We penalize this heavily so the BO avoids it.
        if std_s1 == 0.0 or std_s2 == 0.0:
            set_loss = 100.0  # High penalty
        else:
            # We want to minimize the standard deviation (maximize resolution)
            set_loss = std_s1 + std_s2 
            valid_sets += 1
            
        total_loss += set_loss
        
    # Return average loss across all physical contexts
    return total_loss / len(selected_sets)